In [5]:
import pandas as pd
import numpy as np
from scipy import sparse
from recovery_model.system_builder import System

In [1]:
# from itertools import product, chain
# from collections import defaultdict
# from dataclasses import dataclass
# from operator import mul
# from functools import reduce
# from pprint import pprint, pformat
# from typing import Literal
# import json

In [2]:
composition_dct = {
    "url": "data/basic/Toy_WEEE_Composition.xlsx",  # "../../data/WEE/Toy_WEEE_Composition.xlsx",
    "sheet": {
        "P_C": {
            "index": ("Flow", "Category", "Component"),
            "columns": ("Flow", "Category"),
            "data": "Share_Component",
        },
        "P_M": {
            "index": ("Flow", "Category", "Material"),
            "columns": ("Flow", "Category"),
            "data": "Share_Material",
        },
        "PC_E": {
            "index": ("Flow", "Category", "Component", "Element"),
            "columns": ("Flow", "Category", "Component"),
            "data": "Share_element",
        },
        "PM_E": {
            "index": ("Flow", "Category", "Material", "Element"),
            "columns": ("Flow", "Category", "Material"),
            "data": "Share_element",
        },
    },
    "mapper": {
        "Flow": "flows",
        "Category": "products",
        "Component": "components",
        "Material": "materials",
        "Element": "elements",
        "Share_Component": "data",
        "Share_Material": "data",
        "Share_element": "data",
        "Year": "year",
    },
}

tc_dct = {
    "url": "data/basic/Toy_WEEE_TCs.xlsx",  #  "../../data/WEE/Toy_WEEE_TCs.xlsx",
    "crossboundary_inflows": ("WEEE_INFLOW",),
    "sheet": {
        "Toy_WEEE_TCs": {
            "flows": ("input_flow", "output_flow"),
            "data": "value",
        },
    },
    "mapper": {
        "value": "data",
        "Year": "year",
    },
}

inputs_dct = {
    "url": "data/basic/Toy_WEEE_inputs.xlsx",  # "../../data/WEE/Toy_WEEE_inputs.xlsx",
    "sheet": {
        "Toy_WEEE_inputs": {
            "index": ("input_flow", "Category"),
            "columns": None,
            "data": "mass_(tonnes)",
        },
    },
    "unit": "tonnes",
    "mapper": {
        "input_flow": "flows",
        "Category": "products",
        "mass_(tonnes)": "data",
        "Year": "year",
    },
}

To do:

- set ones on diagonal
- set y (mass)
- solve and validate


In [3]:
# @dataclass
# class System:
#     composition_dct: dict
#     inputs_dct: dict
#     tc_dct: dict
#     __all_keys = ("flows", "products", "components", "materials", "elements")  # first = "flows" !
#     __idx_keys = ("flows", "products", "components", "materials")  # first = "flows" !
#     __col_keys = ("products", "components", "materials", "elements")

#     def __post_init__(self, fillna=-1):
#         self.__init_var()
#         self.__set_index(fillna=fillna)
#         self.__set_columns()
#         # rows, cols, data = self.__input_mass(fillna=fillna)
#         # self.__set_y(rows, cols, data)
#         # rows, cols, data = self.__input_composition(fillna=fillna)
#         rows, cols, data = self.__read_input(dct=self.composition_dct, fillna=-1)
#         self.__init_input_tcs(rows, cols, data)
#         rows, cols, data = self.__read_input(dct=self.inputs_dct, fillna=-1)
#         self.__init_y(rows, cols, data)

#         # rows, cols, data = self.__read_inflow()
#         # self.__init_y(rows, cols, data)
#         # rows, cols, data = self.__read_tcs()
#         # self.__init_tcs(rows, cols, data)
#         # self.__set_metadata()

#     def __init_var(self):
#         keys = self.__all_keys
#         self.__var = dict()
#         # get flow names from excel file
#         flow_names = set()
#         filepath = self.tc_dct["url"]
#         file = pd.read_excel(filepath, sheet_name=None)
#         for sheet in file.keys():
#             for flow in self.tc_dct["sheet"][sheet]["flows"]:
#                 new_flow_names = file[sheet][flow].unique()
#                 flow_names.update(new_flow_names)
#         self.__var.update({keys[0]: tuple(sorted(flow_names))})  # ! order of attrs matters
#         # get variable names (products, materials, components and elements) from excel file
#         variables = {name: set() for name in keys[1:]}
#         filepath = self.composition_dct["url"]
#         file = pd.read_excel(filepath, sheet_name=None)
#         mapper = self.composition_dct["mapper"]  # to ensure consistent naming
#         for sheet in file.keys():
#             df = file[sheet].rename(mapper=mapper, axis=1)  # apply consistent naming
#             for col in set(variables.keys()) & set(df.columns):
#                 new_variables = df[col].unique()
#                 (variables[col]).update(new_variables)
#         self.__var.update({k: tuple(sorted(v)) for k, v in variables.items()})

#     def __set_index(self, fillna):
#         idxs = self.__idx_keys
#         # fillna to allow direct links to sublevels
#         iterables = [self.__var[idxs[0]]] + [[fillna] + list(self.__var[i]) for i in idxs[1:]]
#         self.__index = pd.MultiIndex.from_product(iterables, names=idxs)
#         self.__var["index"] = {
#             idxs[i]: {v: k for k, v in enumerate(seq)} for i, seq in enumerate(iterables)
#         }

#     def __set_columns(self):
#         cols = self.__col_keys
#         iterables = (product([name], self.__var[name]) for name in cols)
#         self.__columns = pd.MultiIndex.from_tuples(tuple(chain.from_iterable(iterables)))
#         self.__var["columns"] = {v: k for k, v in enumerate(self.__columns)}

#     def __input_composition(self, fillna, year=None):
#         # Specify sheet, and columns we are interested (values need to be sequences)
#         rows, cols, data = [], [], []
#         filepath = self.composition_dct["url"]
#         mapper = self.composition_dct["mapper"]
#         file = pd.read_excel(filepath, sheet_name=None)
#         for sheet in file.keys():
#             df = file[sheet].rename(mapper=mapper, axis=1)
#             # get names of excel columns to be fetched and allocate them to corresponding label category
#             row_labels = [mapper[i] for i in self.composition_dct["sheet"][sheet]["index"]]
#             col_labels = [mapper[i] for i in self.composition_dct["sheet"][sheet]["columns"]]
#             data_labels = [mapper[i] for i in self.composition_dct["sheet"][sheet]["data"]]
#             # filter if necessary
#             mask = df["year"] == year if year else slice(None)
#             # get appropriate data column
#             data.append(df.loc[mask, data_labels].values)  # size = n x 1
#             # rows needs to be broadcasted to size nx4 (flows + products + materials + elements)
#             #     - first set multiindex as an nx4 array filled with fillna (here = -1)
#             midx_data = np.full(shape=(len(df[mask]), len(self.__idx_keys)), fill_value=fillna)
#             multiidx = pd.DataFrame(data=midx_data, columns=self.__idx_keys)
#             #     - then update -1 with the actual flows / products / materials / elements values we have
#             multiidx.loc[:, row_labels] = df[row_labels].values
#             rows.append(multiidx.values)
#             # columns need to be broadcasted to size nx2
#             # this is done in case same name is used accross comp / mat / elt
#             # (e.g. aluminium as alloy and chemical element)
#             for label in col_labels:
#                 cols_lvl1 = df.loc[mask, label].values
#                 cols_lvl0 = np.full_like(cols_lvl1, fill_value=label)
#                 cols.append(np.vstack([cols_lvl0, cols_lvl1]).T)
#         print(data)
#         return (np.vstack(rows), np.vstack(cols), np.vstack(data))

#     def broadcast_idxs(self, df_idx, fillna):
#         # rows needs to be broadcasted to size nx4 (flows + products + materials + elements)
#         #    - first set multiindex as an nx4 array filled with fillna (here = -1)
#         #    - then update -1 with the actual flows / products / materials / elements values we have
#         missing_cols = list(set(self.__idx_keys) - set(df_idx.columns))
#         new_idx = df_idx.copy()
#         new_idx[missing_cols] = np.full(shape=(len(new_idx), len(missing_cols)), fill_value=fillna)
#         return new_idx.loc[:, self.__idx_keys].values

#     def broadcast_col(self, df_col, label):
#         # columns need to be broadcasted to size nx2
#         # this is done in case same name is used accross comp / mat / elt
#         # (e.g. aluminium as alloy and chemical element)
#         cols_lvl1 = df_col.values
#         cols_lvl0 = np.full_like(cols_lvl1, fill_value=label)
#         return np.vstack([cols_lvl0, cols_lvl1]).T

#     def __read_input(self, dct, fillna, year=None):
#         # Specify sheet, and columns we are interested (values need to be sequences)
#         rows, cols, data = [], [], []
#         filepath = dct["url"]
#         mapper = dct["mapper"]
#         excel = dct["sheet"]
#         file = pd.read_excel(filepath, sheet_name=None)
#         for sheet in file.keys():
#             df = file[sheet].rename(mapper=mapper, axis=1)
#             # filter if necessary
#             mask = df["year"] == year if year else slice(None)
#             # get names of system' rows (= Systems.index) from corresponding excel columns
#             row_labels = [mapper[i] for i in excel[sheet]["index"]]
#             # expand rows to nx4 shape (flows, products, components, materials)
#             expanded_idx = self.broadcast_idxs(df.loc[mask, row_labels], fillna)
#             rows.append(expanded_idx)  # size = n x 4
#             # get names of system' columns (= Systems.columns) from corresponding excel column
#             col_label = mapper[excel[sheet]["column"]]
#             # expand column to mx2 shape
#             expanded_cols = self.broadcast_col(df.loc[mask, col_label], col_label)
#             cols.append(expanded_cols)  # size = m x 2
#             # get data from corresponding excel column
#             data_label = mapper[excel[sheet]["data"]]
#             data.append(df.loc[mask, data_label].values)  # size = n x 1
#         return (np.vstack(rows), np.vstack(cols), np.hstack(data))

#     def __map_arr(self, arr, mapper):
#         return np.vectorize(mapper.__getitem__)(arr)

#     def __map_to_midx(self, arr, mapper):
#         keys = self.__idx_keys
#         res = [self.__map_arr(arr[:, i], mapper[keys[i]]) for i in range(arr.shape[1])]
#         return np.vstack(res).T

#     def __midx_to_iloc(self, target, shape):
#         # target needs to be tup[int]
#         # coeffs = (a0, a1, ..., an-1, an), with:
#         #   a0 = shape[1] * shape[2] * ... * shape[n] * 1,
#         #   a1 = shape[2] * ... * shape[n] * 1,
#         #   an-1 = shape[n]
#         #   an = 1
#         coeff = np.array(
#             [reduce(mul, shape[i + 1 :]) if i + 1 < len(shape) else 1 for i, _ in enumerate(shape)]
#         )
#         try:
#             return (coeff * target).sum(axis=1)
#         except np.AxisError:  # if target was a 1d array
#             return (coeff * target).sum()

#     def __init_input_tcs(self, rows, cols, data):
#         # convert rows + cols from str to ndarray[int]
#         cols_iloc = np.array([self.__var["columns"][i] for i in zip(cols[:, 0], cols[:, 1])])
#         # (flatten idxs requires 2 steps)
#         rows_as_int = self.__map_to_midx(arr=rows, mapper=self.__var["index"])
#         rows_iloc = self.__midx_to_iloc(rows_as_int, self.index.levshape)
#         # data needs to be squeeze to also become 1d
#         # coo_mat = sparse.coo_matrix(
#         #     (data, (rows_iloc, cols_iloc)),
#         #     shape=(len(self.index), len(self.columns)),  # ! temporary index
#         #     # shape=(len(self.index), len(self.flows)),
#         # )
#         # ! TEST
#         coo_mat = sparse.coo_matrix(
#             (data, (rows_iloc, rows_iloc)),
#             shape=(len(self.index), len(self.index)),  # ! temporary index
#             # shape=(len(self.index), len(self.flows)),
#         )
#         self.__tcs = coo_mat
#         # self.__composition = pd.DataFrame.sparse.from_spmatrix(
#         #     data=coo_arr.tocsr(),  # convert to Compressed Sparse Row matrix
#         #     index=self.index,
#         #     columns=self.columns,
#         # )
#         # ! to be completed (add ones on diagonal)
#         # self.__mass = data
#         # self.__composition = rows_as_int

#     def __init_y(self, rows, cols, data):
#         cols_iloc = np.array([self.__var["columns"][i] for i in zip(cols[:, 0], cols[:, 1])])
#         rows_as_int = self.__map_to_midx(arr=rows, mapper=self.__var["index"])
#         rows_iloc = self.__midx_to_iloc(rows_as_int, self.index.levshape)
#         coo_mat = sparse.coo_matrix(
#             (data, (rows_iloc, cols_iloc)),
#             shape=(len(self.index), len(self.columns)),  # ! temporary
#             # shape=(len(self.index), len(self.flows)),
#         )
#         self.__y = coo_mat

#     def __read_inflow(self, inputs=150):
#         nb_cols = len(self.__columns)
#         nb_inner_rows = len(self.__index) // len(self.__flow_names)
#         rows = np.random.randint(nb_inner_rows, size=inputs)
#         cols = np.random.randint(nb_cols, size=inputs)
#         data = np.random.randint(1, 10, size=inputs)
#         return (rows, cols, data)

#     def __read_tcs(self, inputs=150):
#         return (0, 1, 2)

#     def __set_metadata(self):
#         self.__output = dict()
#         self.__output["type"] = "composition"
#         self.__output["composition"] = self.__composition
#         self.__output["mass"] = self.__mass

#     @property
#     def flows(self):
#         return self.__var["flows"]

#     @property
#     def products(self):
#         return self.__var["products"]

#     @property
#     def components(self):
#         return self.__var["components"]

#     @property
#     def materials(self):
#         return self.__var["materials"]

#     @property
#     def elements(self):
#         return self.__var["elements"]

#     @property
#     def index(self):
#         return self.__index

#     @property
#     def columns(self):
#         return self.__columns

#     # @property
#     # def output(self):
#     #     return self.__output["type"]

#     # @output.setter
#     # def output(self, name):
#     #     assert name in {"composition", "mass"}, "options are ('composition', 'mass')"
#     #     self.__output["type"] = name

#     @property
#     def inflow(self):
#         # return self.__composition.loc[self.__flow_names[0]]
#         return self.__inflow

#     @property
#     def rows(self):
#         return self.__rows

#     @property
#     def cols(self):
#         return self.__cols

#     @property
#     def data(self):
#         return self.__data

#     def __str__(self):
#         # return json.dumps(self.__var, indent=4)
#         return pformat(self.__var, indent=4)

#     @property
#     def var(self):
#         return self.__var

#     @property
#     def y(self):
#         # return self.__output[self.__output["type"]]
#         return self.__y

#     def __getitem__(self, items):
#         return self.y.loc[items]

In [3]:
# # ------------------------------------------------------------
# # UTILITY FUNCTION
# # ------------------------------------------------------------


# def map_1Darray(arr, mapper):
#     """Map the elements of an 1D array using a mapper dictionnary.

#     Args:
#         arr (ndarray): The input array.
#         mapper (dict): A dictionary mapping the elements of the array to their corresponding values.

#     Returns:
#         ndarray: The mapped 1D array.
#     """
#     return np.vectorize(mapper.__getitem__)(arr)


# def map_2Darray(arr, mapper, keys):
#     """Map the elements of a 2D array using a mapper dictionnary.

#     Args:
#         arr (ndarray): The input 2D array.
#         mapper (dict): A dictionary mapping the elements of each column to their corresponding values.
#         keys (list): The list of keys representing the columns of the array (in the right order)

#     Returns:
#         ndarray: The mapped 2D array.
#     """
#     res = [map_1Darray(arr=arr[:, i], mapper=mapper[keys[i]]) for i in range(arr.shape[1])]
#     return np.vstack(res).T


# def map_array(arr, mapper, keys):
#     """
#     Map the elements of a 1D or 2D array using a mapper dictionnary.

#     Args:
#         arr (numpy.ndarray): The input array to be mapped.
#         mapper (dictionnary): The dictionnary to be applied to each element of the array.
#         keys (list): The list of keys to be used for mapping the elements of the array (in the right order).

#     Raises:
#         ValueError: If the input array is not 1D or 2D.

#     Returns:
#         numpy.ndarray: The mapped array.
#     """
#     if arr.ndim == 1:
#         return map_1Darray(arr, mapper)
#     elif arr.ndim == 2:
#         return map_2Darray(arr, mapper, keys)
#     else:
#         raise ValueError("Array must be 1D or 2D")


# def map_multiindx_to_iloc(arr, shape):
#     """Convert a list of points in a multiindex to their corresponding integer-based indices.

#     Args:
#         arr (ndarray[int]): Array of integers representing the location along each dimension of multiple points
#         shape (tuple): The dimensional space

#     Returns:
#         ndarray: The integer-based indices = (a0, a1, ..., an-1, an), with:
#             a0 = shape[1] * shape[2] * ... * shape[n] * 1,
#             a1 = shape[2] * ... * shape[n] * 1,
#             an-1 = shape[n]
#             an = 1
#     """
#     coeff = np.array([reduce(mul, shape[i + 1 :]) if i + 1 < len(shape) else 1 for i, _ in enumerate(shape)])
#     try:
#         return (coeff * arr).sum(axis=1)
#     except np.AxisError:  # if target was a 1d array
#         return (coeff * arr).sum()


# # ------------------------------------------------------------
# # CLASS METHODS
# # ------------------------------------------------------------


# @dataclass
# class System:
#     composition_dct: dict
#     inputs_dct: dict
#     tc_dct: dict
#     __idx_keys = ("flows", "products", "components", "materials", "elements")  # "flows" comes 1st

#     # ------------------------------------------------------------
#     # SYSTEM INITIALIZATION
#     # ------------------------------------------------------------

#     def __post_init__(self, fill_idx=-1):
#         """Initialize the System class.

#         Args:
#             fill_idx [int or str]: The value that represents the bypassing of a hierarchical level (default = -1)
#             e.g. [F1, P1, -1, M1] represents the mass fraction of M1 in P1 (in F1)
#             which is NOT part of C1, C2, etc.
#             fill_idx does NOT apply to flows (only products, components, materials and elements)
#         """
#         self.__get_var_names()
#         self.__set_index(fill_idx=fill_idx)

#         # ! 1) Process composition data
#         comp_data, comp_rows, comp_cols = self.__read_input(dct=self.composition_dct, fill_idx=fill_idx)

#         comp_data = -comp_data
#         comp_rows = self.get_indexer(targets=comp_rows)
#         comp_cols = self.get_indexer(targets=comp_cols)

#         # ! 2) Process transfer coefficients data

#         # ! 3) Process input data (mass of flows entering the system
#         input_data, input_rows, _ = self.__read_input(dct=self.inputs_dct, fill_idx=fill_idx)
#         input_rows = self.get_indexer(targets=input_rows)

#         # ! 4) Create the matrix of TCs
#         composition = (comp_data, comp_rows, comp_cols)
#         flow_tcs = None
#         self.__fill_tcs(composition=composition, flow_tcs=flow_tcs)

#         # ! 5) Create the Y vector
#         self.__fill_y(data=input_data, rows=input_rows)

#     def __get_var_names(self):
#         """Get variable names (products, components, materials, elements) from composition excel file."""
#         self.__var = dict()

#         # ! 1) Get flow names from transfer coefficients excel file
#         flow_names = set()
#         # get the path to the excel file with the transfer coefficients
#         filepath = self.tc_dct["url"]
#         # get the file sheet names
#         excel = self.tc_dct["sheet"]
#         # read the excel file
#         file = pd.read_excel(filepath, sheet_name=None)
#         # get the unique flow names from the "flows" columns of each sheet
#         for sheet in file.keys():
#             for flow in excel[sheet]["flows"]:
#                 new_flow_names = file[sheet][flow].unique()
#                 flow_names.update(new_flow_names)
#         # sort and store the flow names
#         self.__var.update({self.__idx_keys[0]: tuple(sorted(flow_names))})

#         # ! 2) Get variable names (products, components, materials and elements) from composition excel file
#         variable_names = {name: set() for name in self.__idx_keys[1:]}
#         filepath = self.composition_dct["url"]
#         mapper = self.composition_dct["mapper"]
#         file = pd.read_excel(filepath, sheet_name=None)
#         for sheet in file.keys():
#             # Rename columns from excel file to ensure consistent naming across waste streams
#             df = file[sheet].rename(mapper=mapper, axis=1)
#             # Only consider columns that are either products, components, materials or elements
#             for col in set(variable_names.keys()) & set(df.columns):
#                 new_variables = df[col].unique()
#                 (variable_names[col]).update(new_variables)
#         self.__var.update({k: tuple(sorted(v)) for k, v in variable_names.items()})

#         # 3) Get unit of input data
#         self.__var["unit"] = self.inputs_dct["unit"]

#     def __set_index(self, fill_idx):
#         """Set the index of the System.

#         Args:
#             fill_idx [int or str]: The value to bypass the hierarchical decomposition.
#         """
#         idxs = self.__idx_keys
#         iterables = [self.__var[idxs[0]]] + [[fill_idx] + list(self.__var[i]) for i in idxs[1:]]
#         # store index as pd.DataFrame for easier manipulation (see __getitem__)
#         __index = pd.MultiIndex.from_product(iterables, names=idxs)
#         self.__index = __index.to_frame()
#         # also store values of each multiindex levels for printing (see __str__)
#         self.__var["index"] = {idxs[i]: {v: k for k, v in enumerate(seq)} for i, seq in enumerate(iterables)}

#     def __broadcast_idxs(self, df_idx, fill_idx):
#         """Broadcast partial index into complete index (e.g. [F1, M1, E1] --> [F1, -1, -1, M1, E1])

#         Args:
#             df_idx (pandas.DataFrame): The partial index to be broadcasted.
#             fill_idx (int or str): The value to fill the missing columns with.

#         Returns:
#             numpy.ndarray: The complete index with the missing columns filled.

#         """
#         if isinstance(df_idx, pd.Series):
#             df_idx = df_idx.to_frame()
#         missing_cols = list(set(self.__idx_keys).difference(set(df_idx.columns)))
#         new_idx = df_idx.copy()
#         new_idx[missing_cols] = np.full(shape=(len(new_idx), len(missing_cols)), fill_value=fill_idx)
#         return new_idx.loc[:, self.__idx_keys]  # Sort columns in the right order

#     def __read_input(self, dct, fill_idx, year=None):
#         """Read input data from an Excel file and process it.

#         Args:
#             dct (dict): A dictionary containing the file path, mapper, and sheet information.
#             fill_idx (str): The value to fill missing data with.
#             year (int, optional): The year to filter the data by. Defaults to None.

#         Returns:
#             tuple: A tuple containing three arrays: rows, cols, and data.

#         """
#         # Specify sheet, and columns we are interested (values need to be sequences)
#         rows, cols, data = [], [], []
#         filepath = dct["url"]
#         mapper = dct["mapper"]
#         excel = dct["sheet"]
#         file = pd.read_excel(filepath, sheet_name=None)

#         for sheet in file.keys():
#             df = file[sheet].rename(mapper=mapper, axis=1)
#             # Filter the data if necessary
#             mask = df["year"] == year if year else slice(None)

#             # 1) Get names of system's rows (= Systems.index) from corresponding excel columns
#             row_labels = [mapper[i] for i in excel[sheet]["index"]]
#             # Expand rows to nx5 shape (flows, products, components, materials, elements)
#             expanded_idx = self.__broadcast_idxs(df.loc[mask, row_labels], fill_idx)
#             rows.append(expanded_idx.values)  # size = n x 4

#             # 2) Get names of system's columns (= Systems.columns) from corresponding excel column
#             if excel[sheet]["columns"] is not None:
#                 col_label = [mapper[i] for i in excel[sheet]["columns"]]
#                 # Expand column to mx2 shape
#                 expanded_cols = self.__broadcast_idxs(df.loc[mask, col_label], fill_idx)
#                 cols.append(expanded_cols.values)  # size = m x 2

#             # 3) Get data from corresponding excel column
#             data_label = mapper[excel[sheet]["data"]]
#             data.append(df.loc[mask, data_label].values)  # size = n x 1

#         data = np.hstack(data)
#         rows = np.vstack(rows)
#         cols = np.vstack(cols) if cols != [] else None
#         return (data, rows, cols)

#     def __fill_tcs(self, composition, flow_tcs):
#         """Create the matrix of TCs (transfer coefficient) as a sparse matrix.

#         Args:
#             composition (tuple): A tuple containing 3 arrays: data, rows and cols.
#             flow_tcs (tuple): A tuple containing 3 arrays: data, rows and cols.
#         """
#         comp_data, comp_rows, comp_cols = composition
#         # tcs_data, tcs_rows, tcs_cols = flow_tcs  # ! TO BE IMPLEMENTED

#         # add ones on the diagonal
#         diag_data = np.ones(len(self.index))
#         diag_idxs = np.arange(len(self.index))

#         # combine data, rows, cols
#         data = np.hstack([comp_data, diag_data])
#         rows = np.hstack([comp_rows, diag_idxs])
#         cols = np.hstack([comp_cols, diag_idxs])

#         coo_mat = sparse.coo_matrix((data, (rows, cols)), shape=(len(self.index), len(self.index)))
#         csr_mat = coo_mat.tocsr()
#         self.__tcs = csr_mat

#     def __fill_y(self, data, rows):
#         """Create the vector of constant terms (Y) as a sparse matrix.

#         Args:
#             data (ndarray): The data values of the Y vector.
#             rows (ndarray): The rows of the Y vector.
#         """
#         cols = np.zeros_like(rows)
#         coo_arr = sparse.coo_array((data, (rows, cols)), shape=(len(self.index), 1))
#         csc_arr = coo_arr.tocsc()
#         self.__y = csc_arr

#     # ------------------------------------------------------------
#     # SYSTEM SOLVER
#     # ------------------------------------------------------------

#     def solve(self, output="mass"):
#         """Solve the system of linear equations.

#         Args:
#             output (str, optional): Either in mass or mass fraction. Defaults to "mass".

#         Returns:
#             pd.Series: The solution of the system of linear equations as a pandas Series object.
#         """
#         solution = sparse.linalg.spsolve(self.tcs, self.y)
#         if output == "mass":
#             unit = self.var["unit"]
#             return pd.Series(solution, index=self.index, name=f"mass ({unit})")
#         elif output == "fraction":
#             unit = "%"
#             # ! TO BE IMPLEMENTED

#     # ------------------------------------------------------------
#     # GETTERS AND SETTERS
#     # ------------------------------------------------------------

#     def get_indexer(self, targets):
#         """Get the integer-based indices corresponding to the given targets.

#         Args:
#             targets (ndarray): The targets for which to retrieve the indices.

#         Returns:
#             ndarray: The integer-based indices corresponding to the given targets.
#         """
#         # format targets into appropriate index values
#         midx = self.index_loc(targets)
#         # Convert from 2D-ndarray[str] to 2D-ndarray[int]
#         midx_as_int = map_array(arr=midx, mapper=self.__var["index"], keys=self.__idx_keys)
#         # Convert from 2D-ndarray[int] to 1D-ndarray[int] using corresponding integer-based indices
#         midx_iloc = map_multiindx_to_iloc(arr=midx_as_int, shape=self.index.levshape)
#         return midx_iloc

#     def index_loc(self, targets):
#         """Format targets into appropriate index values

#         Args:
#             targets (Union[pd.IndexSlice, Tuple[Any, ...], np.ndarray]): The targets to retrieve values for.

#         Returns:
#             np.ndarray: The formated index corresponding to the targets
#         """
#         try:  # handle case where targets is a pd.IndexSlice
#             return self.__index.loc[targets].values
#         except KeyError:  # handle case where targets is a single tuple
#             targets = [targets]
#             return self.__index.loc[targets].values
#         except ValueError:  # handle case where targets is a 2D ndarray
#             return targets

#     def index_iloc(self, targets):
#         """Return the index values of the DataFrame at the specified integer-based positions.

#         Args:
#             targets (Union[int, List[int]]): The integer-based positions.

#         Returns:
#             pandas.Index: The index values at the specified positions.
#         """
#         return self.__index.iloc[targets].index

#     @property
#     def var(self):
#         """Get the variable names of the system.

#         Returns:
#             dict: The variables of the system.
#         """
#         return self.__var

#     @property
#     def flows(self):
#         """Get the list of flows within the system.

#         Returns:
#             tuple: The flows' names.
#         """
#         return self.__var["flows"]

#     @property
#     def products(self):
#         """Get the list of products within the system.

#         Returns:
#             tuple: The products' names.
#         """
#         return self.__var["products"]

#     @property
#     def components(self):
#         """Get the list of components within the system.

#         Returns:
#             tuple: The components' names.
#         """
#         return self.__var["components"]

#     @property
#     def materials(self):
#         """Get the list of materials within the system.

#         Returns:
#             tuple: The materials' names.
#         """
#         return self.__var["materials"]

#     @property
#     def elements(self):
#         """Get the list of elements within the system.

#         Returns:
#             tuple: The elements' names.
#         """
#         return self.__var["elements"]

#     @property
#     def index(self):
#         """Get the index of the system.

#         Returns:
#             pd.MultiIndex: The index of the system.
#         """
#         return self.__index.index

#     @property
#     def tcs(self):
#         """Get the transfer coefficients (TCs) matrix of the system.

#         Returns:
#             csr_matrix: The transfer coefficients (TCs) as a Compressed Sparse Rows matrix
#         """
#         return self.__tcs

#     @property
#     def y(self):
#         """Get the constant terms that the linear equations should satisfy

#         Returns:
#             csc_array: constant terms Y as a Compressed Sparse Column array
#         """
#         return self.__y

#     def __str__(self):
#         """Return a string representation of the system.

#         Returns:
#             str: The string representation of the system.
#         """
#         return pformat(self.__var, indent=4)

In [3]:
weee = System(composition_dct=composition_dct, inputs_dct=inputs_dct, tc_dct=tc_dct)

In [6]:
idx = pd.IndexSlice["WEEE_INFLOW", "Cat_1", "Cables", :, :]
weee.get_indexer(idx)

array([240, 241, 242, 243, 244, 245])

In [7]:
weee.index_loc(idx)

array([['WEEE_INFLOW', 'Cat_1', 'Cables', -1, -1],
       ['WEEE_INFLOW', 'Cat_1', 'Cables', -1, 'Al'],
       ['WEEE_INFLOW', 'Cat_1', 'Cables', -1, 'Cu'],
       ['WEEE_INFLOW', 'Cat_1', 'Cables', 'AluminiumAlloyUnspecified',
        -1],
       ['WEEE_INFLOW', 'Cat_1', 'Cables', 'AluminiumAlloyUnspecified',
        'Al'],
       ['WEEE_INFLOW', 'Cat_1', 'Cables', 'AluminiumAlloyUnspecified',
        'Cu']], dtype=object)

In [8]:
idx = ["WEEE_INFLOW", "Cat_1", "Cables", -1, -1]
weee.get_indexer(idx)

array([240])

In [9]:
idx = [["WEEE_INFLOW", "Cat_1", "Cables", -1, -1], ["WEEE_INFLOW", "Cat_2", "Cables", -1, -1]]
weee.get_indexer(idx)

array([240, 258])

In [10]:
intidx = [240, 241, 242, 243, 244, 245]
weee.index_iloc(intidx)

MultiIndex([('WEEE_INFLOW', 'Cat_1', 'Cables', ...),
            ('WEEE_INFLOW', 'Cat_1', 'Cables', ...),
            ('WEEE_INFLOW', 'Cat_1', 'Cables', ...),
            ('WEEE_INFLOW', 'Cat_1', 'Cables', ...),
            ('WEEE_INFLOW', 'Cat_1', 'Cables', ...),
            ('WEEE_INFLOW', 'Cat_1', 'Cables', ...)],
           names=['flows', 'products', 'components', 'materials', 'elements'])

In [12]:
solution = weee.solve()
solution.to_csv("results/solution.csv")
solution

flows                             products  components      materials                  elements
WEEE_2RM_mechRec1Smelter_CuScrap  -1        -1              -1                         -1          0.0
                                                                                       Al          0.0
                                                                                       Cu          0.0
                                                            AluminiumAlloyUnspecified  -1          0.0
                                                                                       Al          0.0
                                                                                                  ... 
WEEE_wasteBinLandfill             Cat_2     PCBUnspecified  -1                         Al          0.0
                                                                                       Cu          0.0
                                                            AluminiumAlloyUnspec

In [13]:
np.vstack(
    [
        weee.tcs.tocoo().row,
        weee.tcs.tocoo().col,
        # weee.tcs.tocoo().data,
    ]
)

array([[   0,    1,    2, ..., 1023, 1024, 1025],
       [   0,    1,    2, ..., 1023, 1024, 1025]], dtype=int32)

In [14]:
tcs = weee.tcs.copy()
temp_tc = pd.DataFrame(tcs.toarray(), index=weee.index, columns=weee.index)


xbrdy_inflow = tc_dct["crossboundary_inflows"][0]
# for cat in ("Cat_1", "Cat_2", "Cat_3", "Cat_4a", "Cat_4b", "Cat_5", "Cat_6"):
#     idx_ax0 = pd.IndexSlice[xbrdy_inflow, cat, -1, -1]
#     idx_ax1 = pd.IndexSlice["products", cat]
#     temp_tc.loc[idx_ax0, idx_ax1] = 1

idx = pd.IndexSlice[xbrdy_inflow, :, :, :]
temp_tc.loc[idx, idx].to_csv("results/temp_tc.csv")

In [15]:
y = weee.y.copy()
temp_y = pd.DataFrame(y.toarray(), index=weee.index, columns=weee.columns)


temp_y.loc[idx].to_csv("results/temp_y.csv")

AttributeError: 'System' object has no attribute 'columns'

In [ ]:
(temp_tc * temp_y).loc[pd.IndexSlice[xbrdy_inflow, :, :, :]].to_csv("tc_dot_y.csv")

In [ ]:
A = temp_tc.loc[pd.IndexSlice[xbrdy_inflow, :, :, :]]
A.to_numpy().shape

In [ ]:
Y = temp_y.loc[pd.IndexSlice[xbrdy_inflow, :, :, :]]
Y.to_numpy().shape

In [ ]:
file = "data/test/TC Matrix Building v2.xlsx"
tcs = pd.read_excel(file, sheet_name="tc", header=None).values
y = pd.read_excel(file, sheet_name="y", header=None).values

In [ ]:
from scipy.sparse.linalg import spsolve
from scipy.sparse import csr_matrix, csc_matrix

A = csr_matrix(tcs)
Y = csr_matrix(y)
spsolve(A, Y)

In [ ]:
Afull = csr_matrix(tcs)
Afull.nonzero()

In [ ]:
np.vstack([Afull.nonzero(), Afull.data]).T

In [ ]:
def squarify(mat):
    max_dim = max(mat.shape)
    dims = np.array(mat.shape)
    new_shape = np.full(dims.shape, fill_value=max_dim)
    new_mat = np.zeros(new_shape)
    new_mat[: mat.shape[0], : mat.shape[1]] = mat
    return new_mat


new_A = squarify(A.to_numpy())

In [ ]:
res = new_A * Y.to_numpy()
pd.DataFrame(res, index=A.index, columns=Y.columns).to_csv("temp_res.csv")

In [ ]:
xbrdy_inflow = tc_dct["crossboundary_inflows"][0]
pp = pd.DataFrame(np.zeros(shape=(len(weee.index), len(weee.columns))), index=weee.index, columns=weee.columns)

prod = "Cat_1"
comp = slice(None)
mat = -1
idx_ax0 = pd.IndexSlice[xbrdy_inflow, prod, comp, mat]
idx_ax1 = pd.IndexSlice["products", prod]

pp.loc[idx_ax0, idx_ax1]

In [ ]:
def pruned(coo_mat, idxs, cols):
    csr_mat = coo_mat.tocsr()

    rows, _ = csr_mat.nonzero()
    unique_rows = np.sort(np.unique(rows))
    data = csr_mat[unique_rows, :].toarray()
    return pd.DataFrame(data=data, index=idxs[unique_rows], columns=cols)

In [ ]:
pruned(weee.y, weee.index, weee.columns).to_csv("test.csv")

In [ ]:
# i = weee.materials[0]
# i
rw = tuple([[-1, i, -1] for i in weee.materials])
cl = tuple([("materials", i) for i in weee.materials])
rw

<hr>

# Sparse matrices


In [16]:
rows1 = np.random.randint(6, size=10)
cols1 = np.random.randint(6, size=10)
data1 = np.random.randint(10, size=10)

print(np.array([rows1, cols1, data1]))

coo_arr1 = sparse.coo_array((data1, (rows1, cols1)), shape=(10, 6))
print(coo_arr1.shape)
csr_arr1 = coo_arr1.tocsr()
coo_arr1.toarray()

[[4 1 2 4 2 4 5 5 5 3]
 [0 4 0 1 4 0 4 0 3 2]
 [7 0 1 5 4 1 4 2 2 0]]
(10, 6)


array([[0, 0, 0, 0, 0, 0],
       [0, 0, 0, 0, 0, 0],
       [1, 0, 0, 0, 4, 0],
       [0, 0, 0, 0, 0, 0],
       [8, 5, 0, 0, 0, 0],
       [2, 0, 0, 2, 4, 0],
       [0, 0, 0, 0, 0, 0],
       [0, 0, 0, 0, 0, 0],
       [0, 0, 0, 0, 0, 0],
       [0, 0, 0, 0, 0, 0]])

In [17]:
rows2 = np.random.randint(6, size=10)
cols2 = np.random.randint(6, size=10)
data2 = np.random.randint(10, size=10)

print(np.array([rows2, cols2, data2]))

coo_arr2 = sparse.coo_array((data2, (rows2, cols2)), shape=(10, 6))
csr_arr2 = coo_arr2.tocsr()
coo_arr2.toarray()

[[0 1 5 2 4 2 3 2 0 0]
 [4 0 0 0 0 1 5 0 2 0]
 [2 8 9 7 4 5 4 6 0 4]]


array([[ 4,  0,  0,  0,  2,  0],
       [ 8,  0,  0,  0,  0,  0],
       [13,  5,  0,  0,  0,  0],
       [ 0,  0,  0,  0,  0,  4],
       [ 4,  0,  0,  0,  0,  0],
       [ 9,  0,  0,  0,  0,  0],
       [ 0,  0,  0,  0,  0,  0],
       [ 0,  0,  0,  0,  0,  0],
       [ 0,  0,  0,  0,  0,  0],
       [ 0,  0,  0,  0,  0,  0]])

In [18]:
(csr_arr1 @ csr_arr2.T).toarray()

array([[  0,   0,   0,   0,   0,   0,   0,   0,   0,   0],
       [  0,   0,   0,   0,   0,   0,   0,   0,   0,   0],
       [ 12,   8,  13,   0,   4,   9,   0,   0,   0,   0],
       [  0,   0,   0,   0,   0,   0,   0,   0,   0,   0],
       [ 32,  64, 129,   0,  32,  72,   0,   0,   0,   0],
       [ 16,  16,  26,   0,   8,  18,   0,   0,   0,   0],
       [  0,   0,   0,   0,   0,   0,   0,   0,   0,   0],
       [  0,   0,   0,   0,   0,   0,   0,   0,   0,   0],
       [  0,   0,   0,   0,   0,   0,   0,   0,   0,   0],
       [  0,   0,   0,   0,   0,   0,   0,   0,   0,   0]])

In [19]:
type(sparse.vstack([coo_arr1, coo_arr2]))

scipy.sparse._coo.coo_matrix

In [20]:
sparse.vstack([coo_arr1, coo_arr2]).toarray()

array([[ 0,  0,  0,  0,  0,  0],
       [ 0,  0,  0,  0,  0,  0],
       [ 1,  0,  0,  0,  4,  0],
       [ 0,  0,  0,  0,  0,  0],
       [ 8,  5,  0,  0,  0,  0],
       [ 2,  0,  0,  2,  4,  0],
       [ 0,  0,  0,  0,  0,  0],
       [ 0,  0,  0,  0,  0,  0],
       [ 0,  0,  0,  0,  0,  0],
       [ 0,  0,  0,  0,  0,  0],
       [ 4,  0,  0,  0,  2,  0],
       [ 8,  0,  0,  0,  0,  0],
       [13,  5,  0,  0,  0,  0],
       [ 0,  0,  0,  0,  0,  4],
       [ 4,  0,  0,  0,  0,  0],
       [ 9,  0,  0,  0,  0,  0],
       [ 0,  0,  0,  0,  0,  0],
       [ 0,  0,  0,  0,  0,  0],
       [ 0,  0,  0,  0,  0,  0],
       [ 0,  0,  0,  0,  0,  0]])

In [21]:
csr_arr1[[0], :].toarray()

array([[0, 0, 0, 0, 0, 0]])

In [22]:
seq = [csr_arr1] * 5
sparse.vstack(seq)

<50x6 sparse matrix of type '<class 'numpy.int64'>'
	with 45 stored elements in Compressed Sparse Row format>

In [23]:
rows3 = np.random.randint(6, size=10)
cols3 = np.random.randint(6, size=10)
data3 = np.random.randint(10, size=10)

print(np.array([rows3, cols3, data3]))

coo_mat = sparse.coo_matrix((data2, (rows2, cols2)), shape=(10, 6))
csr_mat = coo_mat.tocsr()
csr_mat.toarray()

[[3 4 2 0 2 3 3 0 4 2]
 [3 5 2 1 3 5 2 2 5 0]
 [1 1 1 0 8 2 6 4 1 4]]


array([[ 4,  0,  0,  0,  2,  0],
       [ 8,  0,  0,  0,  0,  0],
       [13,  5,  0,  0,  0,  0],
       [ 0,  0,  0,  0,  0,  4],
       [ 4,  0,  0,  0,  0,  0],
       [ 9,  0,  0,  0,  0,  0],
       [ 0,  0,  0,  0,  0,  0],
       [ 0,  0,  0,  0,  0,  0],
       [ 0,  0,  0,  0,  0,  0],
       [ 0,  0,  0,  0,  0,  0]])

In [24]:
csr_mat

<10x6 sparse matrix of type '<class 'numpy.int64'>'
	with 9 stored elements in Compressed Sparse Row format>

In [25]:
A = np.ones((5, 5))
A

array([[1., 1., 1., 1., 1.],
       [1., 1., 1., 1., 1.],
       [1., 1., 1., 1., 1.],
       [1., 1., 1., 1., 1.],
       [1., 1., 1., 1., 1.]])

In [26]:
S = sparse.csr_matrix(A)
S

<5x5 sparse matrix of type '<class 'numpy.float64'>'
	with 25 stored elements in Compressed Sparse Row format>